# Assignment 3 - K-Means Clustering

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import k_means
from sklearn.preprocessing import StandardScaler
from pathlib import Path

results = Path('results/assignment3/kmeans')
results.mkdir(parents=True, exist_ok=True)

gene_counts = [10, 100, 1000, 5000, 10000]
k_values = [2, 3, 4, 5]

# Load Data

In [8]:
df = pd.read_csv('Data/SRP064298/SRP064298.tsv', sep='\t') # original dataset: change file to 'gene_data.tsv'
metadata = pd.read_csv('Data/SRP064298/metadata_SRP064298.tsv', sep='\t')
metadata = metadata.set_index('refinebio_accession_code')

# Match the metadata to the expression columns.
metadata = metadata.loc[df.columns[1:]]

# Keep only wild-type AL/RF samples, excluding KO samples.
wt_samples = metadata['refinebio_title'].str.startswith(('WT_AL_', 'WT_RF_'))
metadata = metadata.loc[wt_samples].copy()
metadata.index.name = 'sample_id'

# Extract AL or RF from titles such as WT_AL_ZT00_A_RNA.
groups = metadata['refinebio_title'].str.split('_').str[1].rename('group')
# Transpose to samples x genes, then keep the selected samples.
samples = df.drop(columns='Gene').T.loc[metadata.index]
samples.index.name = 'sample_id'

,0,1,2,3,4,5,6,7,8,9,...,41239,41240,41241,41242,41243,41244,41245,41246,41247,41248
sample_id,,,,,,,,,,,,,,,,,,,,,
SRR2534402,17.798948,0.236804,2.256376,2.155539,0.236804,75.965965,12.112131,3.252853,5.750504,3.945687,...,0.236804,2.040154,0.236804,0.236804,0.236804,0.236804,0.236804,1.649079,0.236804,0.236804
SRR2534403,15.751005,0.236517,2.083811,1.777977,0.236517,75.657566,8.097782,3.388784,4.552137,4.162534,...,0.236517,1.920730,1.474749,0.236517,0.236517,0.236517,0.236517,1.554010,0.236517,0.236517
SRR2534404,17.474698,0.237815,1.678759,1.771829,0.237815,75.965965,13.065127,3.316271,5.041619,4.286779,...,1.290812,0.237815,0.237815,0.237815,0.237815,0.237815,0.237815,1.708130,0.237815,0.237815
SRR2534405,17.229629,0.237395,1.801651,1.775136,1.539616,75.965965,10.695516,3.805256,4.775541,4.609605,...,1.755255,0.237395,1.479612,0.237395,0.237395,0.237395,0.237395,1.675913,0.237395,0.237395
SRR2534406,16.963489,0.234727,2.597309,1.752313,0.234727,75.657566,10.135164,3.483624,5.677525,3.567972,...,0.234727,1.487611,0.234727,0.234727,0.234727,0.234727,0.234727,0.234727,0.234727,0.234727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR2535043,19.152004,0.222731,2.823605,0.222731,0.222731,72.313127,8.309649,3.793868,4.213106,4.059145,...,0.222731,1.362912,0.222731,0.222731,0.222731,0.222731,0.222731,0.222731,0.222731,0.222731
SRR2535044,20.567181,0.222293,2.466684,0.222293,0.222293,71.722860,9.043895,3.530956,5.038329,3.923071,...,0.222293,2.011712,0.222293,0.222293,0.222293,0.222293,0.222293,1.753372,0.222293,0.222293
SRR2535045,19.899393,0.225023,2.464955,0.225023,0.225023,72.313127,7.889099,3.835656,4.718733,3.752625,...,0.225023,0.225023,2.136142,0.225023,0.225023,0.225023,0.225023,1.915193,0.225023,0.225023


# Select variable genes

In [9]:
gene_variance = samples.var(axis=0)
gene_order = gene_variance[gene_variance > 0].sort_values(
    ascending=False, kind='mergesort'
).index

gene_ranking = pd.DataFrame({
    'gene_symbol': df['Gene'],
    'input_row': df.index,
    'variance': gene_variance,
}).loc[gene_order]
gene_ranking.index.name = 'gene_row'

for n_genes in gene_counts:
    gene_ranking.iloc[:n_genes].to_csv(results / f'top_{n_genes}_genes.csv')

gene_ranking.head(10)

,gene_symbol,input_row,variance
gene_row,,,
9917,ENSMUSG00000035202,9917,1604.011366
30934,ENSMUSG00000098973,30934,577.956662
29315,ENSMUSG00000096674,29315,562.459708
29326,ENSMUSG00000096688,29326,505.713260
7621,ENSMUSG00000030098,7621,491.777940
19896,ENSMUSG00000073867,19896,476.386073
29454,ENSMUSG00000096921,29454,443.752424
20001,ENSMUSG00000074115,20001,421.673108
9106,ENSMUSG00000032786,9106,387.049713


# Run K-Means Clustering

In [94]:
summaries = []

for n in gene_counts:
    selected_genes = gene_order[:n]
    scaled_data = StandardScaler().fit_transform(samples.loc[:, selected_genes])

    for i, k in enumerate(k_values):
        centroids, labels, inertia = k_means(X=scaled_data, n_clusters=k, random_state=42)
        d = pd.DataFrame({'sample_id': samples.index, 'label': pd.Series(labels)})
        dsg = d.value_counts('label')
        smallest_cluster = dsg.tail(1)
        largest_cluster = dsg.head(1)
        summaries.append({
            'n_genes': n,
            'k': k,
            'smallest_cluster': smallest_cluster.index.array[0],
            'smallest_size': smallest_cluster.array[0],
            'largest_cluster': largest_cluster.index.array[0],
            'largest_size': largest_cluster.array[0],
            'inertia_normal': inertia / n       # inertia score (sum of squared distances from nearest centroid) divided by # of genes
        })
    
summary = pd.DataFrame(summaries)
summary

C:\Users\pokes\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\pokes\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\pokes\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\pokes\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Window

,n_genes,k,smallest_cluster,smallest_size,largest_cluster,largest_size,inertia_normal
0,10,2,1,271,0,275,412.897736
1,10,3,2,72,1,245,348.292373
2,10,4,2,72,1,174,307.031727
3,10,5,2,48,1,155,258.761335
4,100,2,0,219,1,327,452.020137
5,100,3,1,159,2,198,403.123276
6,100,4,0,68,3,166,360.991474
7,100,5,0,69,1,130,345.220351
8,1000,2,0,218,1,328,458.034059
9,1000,3,0,128,1,267,419.588779
